In [30]:
import pandas as pd
import numpy as np
import pdfplumber
from pdfplumber.utils import extract_text
import re

In [15]:
#Looking at AuthorList.pdf it is clear that the toppart is NOT a part of the actual authors on the project. 
#We need to remove these. Notice that there is a difference in fontsize - so let us read the pdf and extract fontsizes. 
#We use pdfplumber and extract all text from the pages and fontsizes. 
with pdfplumber.open("AuthorList.pdf") as pdf:
    all_text = []
    fontsizes = []
    for page in pdf.pages:
        all_text.append(page.extract_text() or "")
        fontsizes.append([char['size'] for char in page.chars])
    full_text = "\n".join(all_text)

In [16]:
print(full_text)
print(len(full_text))

TheAstrophysicalJournalLetters,848:L12(59pp),2017October20 Abbottetal.
Simon,J.D.,Shappee,B.J.,Drout,M.R.,etal.2017,GCN,21551 Troja,E.,Sakamoto,T.,Cenko,S.B.,etal.2016,ApJ,827,102
Singer,L.P.,etal.2017c,GCN,21569 Troja,E.,Watson,A.,Covina,S.,etal.2017b,GCN,21682
Singer,L.P.,Chen,H.-Y,Holz,D.E.,etal.2016,arXiv:1603.07333 Troja,E.,Ricci,R.,Wieringa,M.L.,&Piro,L.2017f,GCN,21803
Singer,L.P.,Lau,R.,Kasliwal,M.M.,etal.2017a,GCN,21552 Tunnicliffe,R.L.,Levan,A.J.,Tanvir,N.R.,etal.2014,MNRAS,437,1495
Singer,L.P.,Lau,R.,Kasliwal,M.M.,etal.2017b,GCN,21779 Ubertini,P.,Lebrun,F.,diCocco,G.,etal.2003,A&A,411,L131
Singer,L.P.,&Price,L.2016,PhRvD,93,024013 Valenti,S.,Yang,S.,Sand,D.,etal.2017,GCN,21606
Singh,K.P.,Tandon,S.N.,Agrawal,P.C.,etal.2014,ASTROSATMission, vanHaarlem,M.P.,Wise,M.W.,Gunst,A.W.,etal.2013,A&A,556,A2
doi:10.1117/12.2062667 Vedrenne,G.,Roques,J.-P.,Schönfelder,V.,etal.2003,A&A,411,L63
Smartt,S.J.,etal.2017,Natur,https://doi.org/10.1038/nature24303 Veitch,J.,Raymond,V.,Farr,B.,etal.

In [17]:
#Now look at the fontsizes: 
all_fontsizes = []
for fonts_per_page in fontsizes:
    all_fontsizes.extend(fonts_per_page)

#Unique font sizes: 
unique_fontsizes = np.unique(np.round(all_fontsizes, 10))
print("Unique font sizes in the document:", unique_fontsizes)
#I.e. we have 5 different fontsizes. Let sort by each and print the text. 

for size in unique_fontsizes:
    print(f"\nText with font size {size}:")
    for page in pdf.pages:
        page_text = ""
        for char in page.chars:
            if np.isclose(char['size'], size):
                page_text += char['text']
        if page_text:
            print(page_text)

Unique font sizes in the document: [ 5.9768  6.9729  7.9702  9.9626 10.4607]

Text with font size 5.9768:
1234567891011121314151617181920212223

Text with font size 6.9729:
1123,45,67891101011121314151617,18192010,21,221223,242526112127281112928,303132,33103473536221037229383914404142,43304445146147483,446153949504751,55223,244753,434654,55561057284611058596014744616215763101,1564,62230,2444281481,6565476610106767386852814130,2469,1973930242117,18707167102821721447312157475,1438,7683952107726,78177179,45060,806681738210393083842832,3314,211128302418523,2433,866446878889392190419165485560308492762572,6765137193655,54,5594,955260,80477796671117,18712897,3598
99165559721005110183301672,189675111027765718321103210429,2464630505622,103322,101055306128441067744522617,47159,23,24151010107,1083879,4726,301091027191034110,9551,79,423,2497,3597,3523,24912615111036465921101047828464710672738,10115151584111575132,33,17443682609159361113684101521523,24163023,2444303991444526,112469085,11325776797,3

In [18]:
# Looking above we can separate the text associated with each fontsize easily. 
# Since we are only interested in the authors, we can choose to only keep text with fontsize 9.9626. 
author_fontsize = unique_fontsizes[3]
group_fontsize = unique_fontsizes[4]

with pdfplumber.open("AuthorList.pdf") as pdf:
    author_text = []
    group_text = []
    for page in pdf.pages:
        author_chars = [ch for ch in page.chars if np.isclose(ch["size"], author_fontsize, atol = 0.5)]
        author_text.append(extract_text(author_chars))
        group_chars = [ch for ch in page.chars if np.isclose(ch["size"], group_fontsize)]
        group_text.append(extract_text(group_chars))

authors = "\n".join(author_text)
groups = "\n".join(group_text)
print("Authors:\n", authors)
print("\n Gruoups \n", groups)

Authors:
 B.P.Abbott , R.Abbott , T.D.Abbott , F.Acernese , K.Ackley , C.Adams , T.Adams , P.Addesso , R.X.Adhikari ,
V. B. Adya , C. Affeldt , M. Afrough , B. Agarwal , M. Agathos , K. Agatsuma , N. Aggarwal , O. D. Aguiar ,
L. Aiello , A. Ain , P. Ajith , B. Allen , G. Allen , A. Allocca , P. A. Altin , A. Amato , A. Ananyeva ,
S. B. Anderson , W. G. Anderson , S. V. Angelova , S. Antier , S. Appert , K. Arai , M. C. Araya , J. S. Areeda ,
N. Arnaud , K. G. Arun , S. Ascenzi , G. Ashton , M. Ast , S. M. Aston , P. Astone , D. V. Atallah ,
P. Aufmuth , C. Aulbert , K. AultONeal , C. Austin , A. Avila-Alvarez , S. Babak , P. Bacon , M. K. M. Bader ,
S. Bae , P. T. Baker , F. Baldaccini , G. Ballardin , S. W. Ballmer , S. Banagiri , J. C. Barayoga , S. E. Barclay ,
B.C.Barish , D.Barker , K.Barkett , F.Barone , B.Barr , L.Barsotti , M.Barsuglia , D.Barta , S.D.Barthelmy ,
J.Bartlett , I.Bartos , R.Bassiri , A.Basti , J.C.Batch , M.Bawaj , J.C.Bayley , M.Bazzan , B.Bécsy ,
C. Beer , M. B

In [19]:
#Extract only group names from group:
group_lines = groups.splitlines()
group_df = pd.DataFrame(group_lines, columns=["Group"])
#Remove any extra commas:
group_df["Group"] = group_df["Group"].str.strip(",")
#Keep only rows that contain at least one letter:
group_df = group_df[group_df["Group"].str.contains(r"[A-Za-z]", regex=True, na=False)]
#Remove rows where the entry is only "and" (ignoring case/whitespace):
group_df = group_df[~group_df["Group"].str.strip().str.fullmatch(r"and", case=False, na=False)]
group_df = group_df.reset_index(drop=True)
group_df

,Group
0,(LIGO Scientific Collaboration and Virgo Colla...
1,(Fermi GBM)
2,(INTEGRAL)
3,(IceCube Collaboration)
4,(AstroSat Cadmium Zinc Telluride Imager Team)
5,(IPN Collaboration)
6,(The Insight-Hxmt Collaboration)
7,(ANTARES Collaboration)
8,(The Swift Collaboration)
9,(AGILE Team)


In [20]:
#We can now seperate each author by commas, and make a dataframe:
def split_outside_parens(text):
    out = []
    buf = []
    depth = 0
    for ch in str(text):
        if ch == "(":
            depth += 1
        elif ch == ")" and depth > 0:
            depth -= 1
        if ch == "," and depth == 0:
            out.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    out.append("".join(buf))
    return out

authors_raw = split_outside_parens(authors.replace("\n", ","))
authors_df = pd.DataFrame(authors_raw, columns=["Author"])
authors_df["Author"] = authors_df["Author"].str.strip()
mask = authors_df["Author"].isin(group_df["Group"])
authors_df_group = authors_df[mask]
man_remov_groups = group_df[~group_df["Group"].isin(authors_df_group["Author"])]
names_to_remove = man_remov_groups["Group"].tolist()
print("Names to remove:", names_to_remove)

authors_df = authors_df[~mask]
authors_df

Names to remove: ['(ePESSTO) ']


,Author
0,B.P.Abbott
1,R.Abbott
2,T.D.Abbott
3,F.Acernese
4,K.Ackley
...,...
4225,O. M. Smirnov
4226,
4227,R. P. Fender
4228,and P. A. Woudt


In [21]:
#Let us manually remove the last group names. 
manual_names = ['(ePESSTO)']
mask_manual = authors_df["Author"].isin(manual_names)
authors_df = authors_df[~mask_manual]
authors_df

,Author
0,B.P.Abbott
1,R.Abbott
2,T.D.Abbott
3,F.Acernese
4,K.Ackley
...,...
4225,O. M. Smirnov
4226,
4227,R. P. Fender
4228,and P. A. Woudt


In [22]:
#Now we can begin to remove "and", and so on.
#If an entry starts with "and", split it into its own row, and also split on newlines:
authors_df["Author"] = (
    authors_df["Author"]
      .str.replace(r"\n+", ",", regex=True)
      .str.strip()
      .str.replace(r"^\s*and\b\s*", "and, ", regex=True)
      .str.split(",")
)
authors_df = authors_df.explode("Author")
authors_df["Author"] = authors_df["Author"].str.strip()
authors_df = authors_df[authors_df["Author"].ne("")]

#Remove rows where the entry is only only numbers:
authors_df = authors_df[~authors_df["Author"].str.fullmatch(r"\d+", na=False)]
#Remove any lone "and"s: 
authors_df = authors_df[~authors_df["Author"].str.fullmatch(r"and", case=False, na=False)]
#Remove spaces after periods:
authors_df["Author"] = authors_df["Author"].str.replace(r"\.\s+", ".", regex=True)
authors_df = authors_df.reset_index(drop=True)
authors_df

C:\Users\asker\AppData\Local\Temp\ipykernel_19604\1564366752.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  authors_df["Author"] = (


,Author
0,B.P.Abbott
1,R.Abbott
2,T.D.Abbott
3,F.Acernese
4,K.Ackley
...,...
3609,S.Makhathini
3610,N.Oozeer
3611,O.M.Smirnov
3612,R.P.Fender


In [29]:
author_list = authors_df["Author"].tolist()
uniq_auth_df = pd.DataFrame(author_list, columns=["Unique Authors"])

def split_last_initials(name):
    text = str(name).strip()
    if " " in text:
        parts = text.split()
        last = parts[-1]
        initials = " ".join(parts[:-1])
    elif "." in text:
        last = text.split(".")[-1]
        initials = text[: -(len(last) + 1)]
    else:
        last = text
        initials = ""
    return pd.Series([last, initials])

uniq_auth_df[["_last", "_initials"]] = uniq_auth_df["Unique Authors"].apply(split_last_initials)
uniq_auth_df = uniq_auth_df.sort_values(["_last", "_initials", "Unique Authors"])
uniq_auth_df = uniq_auth_df.drop(columns=["_last", "_initials"]).reset_index(drop=True)
uniq_auth_df

,Unique Authors
0,A.Aab
1,M.G.Aartsen
2,B.P.Abbott
3,R.Abbott
4,T.D.Abbott
...,...
3609,J.D.Álvarez
3610,R.Šmída
3611,J.Šupík
3612,A.F.Żarnecki


In [27]:
no_of_unique_authors = len(uniq_auth_df)
print(f"Number of unique authors: {no_of_unique_authors}")

Number of unique authors: 3614


## Midpoint author: 

In [28]:
#Sorted: 
median_sort_authors = no_of_unique_authors // 2
median_authors = [uniq_auth_df["Unique Authors"].iloc[median_sort_authors - 1],uniq_auth_df["Unique Authors"].iloc[median_sort_authors]]
print("Median authors:", median_authors)

Median authors: ['K.Link', 'S.D.Linker']
